# GNSS Paper 1 - core pipeline, TRANSFORMER_MODE=bn_only

Runs `run_pipeline.py`: retrains all **14 detectors** (7 tree/instance/shallow-NN classical + RBF-kernel SVM + 6 deep) from scratch, then the full attack suite and stats, end to end:

`01_classical_baseline -> 02_deep_learning_baseline -> 12_operating_point -> 13_blackbox_attacks -> 25_fragility_ci -> 14_multisurrogate_transfer -> 18_full_eval -> 17_latency -> 20_mono -> 19_final_analysis -> 22_mcnemar`

**This is one of two twin notebooks run head-to-head for a direct A/B on the Transformer architecture.** This one sets `TRANSFORMER_MODE=bn_only`: all 9 features collapsed into a single token before embedding (seq_len=1), so self-attention is mathematically a no-op -- the model trains as a deep residual MLP with LayerNorm, not genuine cross-feature attention. The sibling notebook (`kaggle_core_pipeline_bn_tok.ipynb`) sets `TRANSFORMER_MODE=bn_tok` (per-feature tokenization, genuine attention). Both fix the actual bug (collapse to a constant predictor, AUC=0.5, under `MinMaxScaler` -- traced to a missing input-normalization layer that every other deep architecture in this codebase already had). Run both, in parallel on two separate Kaggle accounts, and compare the final numbers to decide which goes in the paper.

This does **not** regenerate the manuscript figures. That is a separate, later step: run `papers/paper1-satnav/make_figures.py` locally against the downloaded tables.

This is also a separate notebook from `kaggle_generalization.ipynb` (experiments 23/24) -- independent, run any time before/after/in parallel.

## Run it overnight WITHOUT losing the result
Use **Save Version -> Save & Run All (Commit)** (top-right), NOT the interactive Run. Expect roughly 1-2h on a P100 -- most of it GridSearchCV in stage 01 and the 3-seed DL training in stage 02; the SVM step uses cuML (GPU, exact RBF kernel) if RAPIDS is present in this Kaggle image, otherwise a scalable CPU approximation, either way it should not dominate the runtime.

**Before you commit, set in the right panel:** Accelerator = **GPU**, Internet = **On**, and **Add Input** = your dataset with `texbat_track_combined.csv`.

## 1. Clone the code (Paper-1 branch)

`Master_Thesis_Part_A` is a **private** repo, so an anonymous clone fails with
`could not read Username for 'https://github.com'`. Before running the next
cell, attach your GitHub token as a Kaggle Secret (NOT pasted into a cell --
a notebook cell can end up shared or public, a Secret cannot):

1. This notebook's right-hand panel -> **Add-ons -> Secrets**.
2. **Add a new secret**: label it `GITHUB_TOKEN`, value = your Personal Access
   Token (fine-grained, scoped to just this repo, `Contents: Read-only` is
   enough to clone).
3. Toggle it **Attached** for this notebook, then save.

The next cell reads it via Kaggle's secrets API at runtime; the token itself
never appears in the notebook source. **Each of the two twin notebooks needs
the secret attached separately** -- Kaggle secrets are per-notebook.

In [ ]:
import os, subprocess, sys

REPO_HOST = "github.com/Ojerinde/Master_Thesis_Part_A.git"
BRANCH    = "paper1-experiment"
DST       = "/kaggle/working/repo"

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    REPO = f"https://{token}@{REPO_HOST}"
    print("Using GITHUB_TOKEN from Kaggle Secrets.")
except Exception as e:
    REPO = f"https://{REPO_HOST}"
    print(f"[WARN] No GITHUB_TOKEN secret found ({e}). Trying an anonymous clone, "
          f"which will fail with 'could not read Username' on a private repo. "
          f"See the markdown cell above to attach one.")

if not os.path.exists(DST):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, DST], check=True)
os.chdir(DST)
print("cwd:", os.getcwd()); print("top-level:", sorted(os.listdir("."))[:20])

## 2. Place the corpus CSV where the loader expects it

In [ ]:
import glob, shutil, os
src = glob.glob("/kaggle/input/**/texbat_track_combined.csv", recursive=True)
assert src, "Attach the Kaggle dataset that contains texbat_track_combined.csv (right panel > Add Input)."
os.makedirs("data/processed", exist_ok=True)
shutil.copy(src[0], "data/processed/texbat_track_combined.csv")
print(f"CSV placed: {os.path.getsize('data/processed/texbat_track_combined.csv'):,} bytes")

## 3. Environment check (do NOT `pip install -r requirements.txt` here)

In [ ]:
import sys, subprocess, importlib, torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU ONLY"))
if not torch.cuda.is_available():
    print("[WARN] No GPU detected. Set Accelerator = GPU in the right panel, then re-run.")
try:
    import cuml
    print("cuML available:", cuml.__version__, "-> SVM will use the GPU, exact RBF kernel.")
except ImportError:
    print("cuML NOT available -> SVM will use the CPU random-Fourier-features approximation.")
for mod, pip_name in [("imblearn","imbalanced-learn"),("xgboost","xgboost"),("lightgbm","lightgbm"),("sklearn","scikit-learn")]:
    try: importlib.import_module(mod)
    except ImportError: subprocess.run([sys.executable,"-m","pip","install","-q",pip_name], check=True)
print("deps OK")

## 4. Run the core pipeline with TRANSFORMER_MODE=bn_only (the ~1-2h GPU job)

Streams every stage's output live so an error surfaces immediately rather than after the fact, AND tees it to `/kaggle/working/pipeline_log_bn_only.txt` -- some numbers the manuscript cites (the Friedman chi-squared statistics, the McNemar pairwise significance count) are only ever printed to the console, never written to a CSV, so this log is the only durable record of them. Fails fast: `run_pipeline.py` stops at the first stage that returns nonzero.

In [ ]:
import os, subprocess, sys, time
env = dict(os.environ, PYTHONPATH=".", PYTHONWARNINGS="ignore", PYTHONUTF8="1",
           TRANSFORMER_MODE="bn_only")
cmd = [sys.executable, "-u", "run_pipeline.py"]
print("running:", " ".join(cmd), "  TRANSFORMER_MODE=bn_only")
t0 = time.time()
p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                     text=True, bufsize=1, encoding="utf-8")
with open("/kaggle/working/pipeline_log_bn_only.txt", "w", encoding="utf-8") as logf:
    for line in p.stdout:
        print(line, end="")
        logf.write(line)
p.wait()
print(f"[pipeline done] exit={p.returncode}  elapsed={(time.time()-t0)/60:.1f} min")
assert p.returncode == 0, f"run_pipeline.py failed with exit {p.returncode} (scroll up for the traceback)."

## 5. Package everything for download (models + tables)

Zips the retrained models and every result table into one archive, named distinctly from the `bn_tok` sibling run so the two don't get mixed up locally.

In [ ]:
import shutil, os
shutil.make_archive("/kaggle/working/results_minmax_bn_only", "zip", "results")
sz = os.path.getsize("/kaggle/working/results_minmax_bn_only.zip")
print(f"Wrote /kaggle/working/results_minmax_bn_only.zip ({sz/1e6:.1f} MB) -- contains results/models and results/tables (no figures; see step 5 note below).")

In [ ]:
import pandas as pd, os
pd.set_option('display.max_rows', None); pd.set_option('display.width', 200)
HEADLINE = [
    'results/tables/adversarial_full_oppoint.csv',   # Table 1 / FGSM,PGD,DLSA,SNA,TPA at the operating point
    'results/tables/blackbox_boundary_all.csv',       # decision-based ASR/median-min-Linf per detector
    'results/tables/blackbox_boundary_ci.csv',        # 95% bootstrap CI on the fragility ranking (Fig 6/7)
    'results/tables/blackbox_boundary_persample.csv', # per-sample min-Linf -> fragility bootstrap CIs
]
for name in HEADLINE:
    if not os.path.exists(name):
        print(f'[missing] {name}'); continue
    g = pd.read_csv(name)
    print('='*70); print(f'{name}  rows={len(g)}'); print('='*70)
    print(g.round(4).to_string(index=False)[:4000])
    print(f'----BEGIN {os.path.basename(name)}----'); print(g.to_csv(index=False)); print(f'----END {os.path.basename(name)}----')
print('Full results in /kaggle/working/results_minmax_bn_only.zip -- download from the committed version Output tab.')
print()
print('Transformer-specific row (the variable under test this run):')
oppoint = pd.read_csv('results/tables/operating_point_recall95.csv')
print(oppoint[oppoint.model == 'Transformer'].to_string(index=False))

### After it finishes
Download from the committed version's **Output** tab and send both back:
- `results_minmax_bn_only.zip` -- unzip into a folder clearly labeled `bn_only` locally, do NOT overwrite the `bn_tok` sibling run's extraction. Contains:
  - `results/models/classical/*.joblib` (including `SVM.joblib`), `results/models/deep_learning/*.pt` -- the 14 retrained detectors, Transformer trained under `TRANSFORMER_MODE=bn_only`.
  - `results/tables/*.csv` -- everything the manuscript's numbers come from. `generalization.csv` is NOT here (that is `kaggle_generalization.ipynb`'s output, separate and independent of this A/B).
- `pipeline_log_bn_only.txt` -- the full console log, including the SVM backend line ("SVM backend: cuml..." or "...CPU, RFF approximation") and the Friedman/McNemar statistics (never written to a CSV).

Compare against the `bn_tok` sibling run's Transformer row and decide which variant goes in the paper before regenerating figures/manuscript numbers from either.